# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #1 — "The Anatomy of Growing Content"
Growth pages (trend up) have a median of 3.2K words and are 184 days old; pages in decline
(trend down) have a median of 2.3K words and are 230 days old, in a large sample size (74.8K
up vs 45.6K down). **Where does the label come from?** `trend_direction`, calculated as 30-day
impression change compared to previous 30-day period, as I used the label `is_declining_label`
in my own capstone project. **My methodology question:** trend direction and page age are not
independent – older pages simply have more 30-day periods where they might have started to go
into decline, so there may be some part of the age difference between two groups that's purely
a function of label timing and not the actual age-driven effect. Indeed, the authors do say that
their result is "directionally robust... even though this remains an observational comparison,"
which is exactly the caveat I'd make – but wouldn't it shrink or even disappear when comparing
only within the same age bands (say, calculating word count difference separately for each age
tier)?

### Finding #2 — “The Freshness Multiplier”
The headline statistic: 365+ day old content refreshed in 30 days experiences a 3.2x improvement in health (10.7 to 34.5), and 57x increase in impressions (71 to 4,039).

**How did they get the name for it?** The name comes from a before-and-after analysis of pages which *were* refreshed in the already small 361+ bucket, which is highlighted elsewhere on the very same page of the paper as being highly unstable ("283:1 only because the sample is tiny and there is just 1 declining page in that bucket").

**Question about my methodology:** The choice of which pages get refreshed is not random,
but rather something that was decided upon – a content team will be more inclined to refresh
pages that they consider to already be important enough to preserve (residual demand, brand
significance, et cetera), so one can argue that the whole effect size of 57x impression boost is,
in part, driven by *which pages got refreshed*, and not how refreshing *itself* affected an old page.
The authors' admission in their paper about the instability of the preceding 283:1 ratio because of
a very small sample size is a perfect example – I'd question whether the refreshed and not-refreshed
subsamples are big enough not to have such small-n bias, and whether the comparison is of refreshed
pages before/after refreshing or between refreshed pages and a control group of similar but not
refreshed pages.

In [4]:
# Framing-only section -- the paper excerpts above are paraphrased from
# docs/flyrank-seo-research-march-2026.pdf (Findings #1 and #4).


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** A simplistic 80/20 row split where the flaw in the methodology that the paper highlights above occurs – putting pages of the same client in both train and test data is great, but it
teaches the model client-specific idiosyncrasies rather than a pattern.


**After:** The client-wise split that I used in W05.

In [1]:
import pandas as pd, numpy as np, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    if not os.path.exists("internship"):
        get_ipython().system('git clone https://github.com/UnzilaAhsan/internship.git')
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

NUM = ["search_volume","competition","cpc","word_count","char_count",
       "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
       "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
       "word_count_tier","impression_tier","position_tier"]

def build_X(frame):
    num = frame[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).copy()
    num["log_impressions_90d"] = np.log1p(frame["impressions_90d"].fillna(0))
    num["log_clicks_90d"]      = np.log1p(frame["clicks_90d"].fillna(0))
    num["log_sessions_90d"]    = np.log1p(frame["sessions_90d"].fillna(0))
    num["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"].fillna(0))
    cat = frame[CAT].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=CAT, dtype=float)
    return pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    idx = np.argsort(-scores)[:k]
    return float(np.asarray(y_true)[idx].mean())

def fit_score(Xtr, ytr, Xte, yte):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(Xtr, ytr)
    s = rf.predict_proba(Xte)[:, 1]
    return {
        "precision@20": precision_at_k(yte, s, 20),
        "precision@50": precision_at_k(yte, s, 50),
        "roc_auc": roc_auc_score(yte, s),
        "avg_precision": average_precision_score(yte, s),
    }

# --- BEFORE: naive random row split ---
X_all, y_all = build_X(df), df["is_declining_label"]
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
before = fit_score(Xtr, ytr, Xte, yte)

# --- AFTER: client-grouped split (same as W05) ---
rng = np.random.default_rng(42)
clients = df["client_id"].unique()
shuffled = rng.permutation(clients)
test_clients = set(shuffled[:max(1, int(round(len(shuffled) * 0.2)))])
test_mask = df["client_id"].isin(test_clients)
train_df, test_df = df[~test_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)
Xtr2_raw, Xte2_raw = build_X(train_df), build_X(test_df)
Xtr2, Xte2 = Xtr2_raw.align(Xte2_raw, join="outer", axis=1, fill_value=0)
after = fit_score(Xtr2, train_df["is_declining_label"], Xte2, test_df["is_declining_label"])

comparison = pd.DataFrame({"BEFORE (random row split)": before, "AFTER (client-grouped split)": after}).T.round(3)
print(comparison)


Cloning into 'internship'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 206 (delta 99), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 1.91 MiB | 1.15 MiB/s, done.
Resolving deltas: 100% (99/99), done.
                              precision@20  precision@50  roc_auc  \
BEFORE (random row split)             0.90          0.94    0.759   
AFTER (client-grouped split)          0.75          0.74    0.751   

                              avg_precision  
BEFORE (random row split)             0.769  
AFTER (client-grouped split)          0.624  


**Reading the before/after**: the random split seems to do *way* better than it deserves
to (precision@20 0.90, avg precision 0.769) compared to the client-based split (precision@20
0.75, avg precision 0.624). This is the kind of difference that results from inflated
performance due to client leakage and not some sort of ability — exactly the type of gap
my paper question on their Finding #4 was addressing. The client-based split (0.75 /
0.624) is the figure I report, the random split is an example of what not to do.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# --- Confirm the label-source fields are excluded from the final feature set ---
final_features = set(X_all.columns)
banned = {"trend_direction", "trend_pct", "impressions_prev_30d", "impressions_last_30d",
          "clicks_prev_30d", "clicks_last_30d", "is_declining_label", "content_id", "client_id"}
overlap = final_features & banned
assert not overlap, f"Leaked fields found in final features: {overlap}"
print(f"Confirmed: none of {sorted(banned)} are in the {len(final_features)}-column final feature set.")

# --- Repeat the deliberate leak-then-remove test from W03, on THIS final feature set ---
leaky_df = X_all.copy()
leaky_df["outcome_impressions_last_30d"] = df["impressions_last_30d"]  # deliberately re-added
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(leaky_df, y_all, test_size=0.2, random_state=42, stratify=y_all)
leak_result = fit_score(Xtr_l, ytr_l, Xte_l, yte_l)
print("\nWITH the deliberate leak (impressions_last_30d) added back on purpose:")
print(pd.Series(leak_result).round(3))
print(f"\nWITHOUT it (this notebook's actual model, random-split before number for comparison):")
print(pd.Series(before).round(3))
print("\nThe leaked-in version's ROC AUC/avg precision jump confirms the same pattern as W03 -- "
      "the actual model above never includes this column, on purpose.")


Confirmed: none of ['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'trend_direction', 'trend_pct'] are in the 52-column final feature set.

WITH the deliberate leak (impressions_last_30d) added back on purpose:
precision@20     1.000
precision@50     1.000
roc_auc          0.816
avg_precision    0.839
dtype: float64

WITHOUT it (this notebook's actual model, random-split before number for comparison):
precision@20     0.900
precision@50     0.940
roc_auc          0.759
avg_precision    0.769
dtype: float64

The leaked-in version's ROC AUC/avg precision jump confirms the same pattern as W03 -- the actual model above never includes this column, on purpose.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

"Random Forest is superior to the baseline on all the metrics."
On this particular data set, under a client-held-out test, the Random Forest model has shown to be more precise@20, more precise@50, have a better ROC AUC, and average precision than the rule-based baseline. It is an observation that is directional in nature, based on one snapshot of one portfolio — it is indicative of the model being a more plausible alternative for the decision than the rule, but does not show whether the model would outperform on another portfolio, another period of time,
or against another baseline rule.

In [3]:
# Framing only for this section.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.